# 01 — Data and protocol

Validate HelpSteer2, prepare the grouped holdout and balanced calibration blocks, and save the run manifest. Complete notebook 00 first. This notebook performs no model inference.

In [ ]:
from pathlib import Path

import pandas as pd

from rubric_order.api import (
    estimate_experiment,
    find_project_root,
    inspect_protocol,
    load_experiment_config,
    prepare_experiment,
    validate_dataset,
)

PROJECT_ROOT = find_project_root(Path.cwd())

In [ ]:
# Keep these values aligned across notebooks.
DATA_PATH = PROJECT_ROOT / "HelpSteer2"
CONFIG_PATH = PROJECT_ROOT / "configs" / "pilot.yaml"
RUN_ID = "pilot-001"
RUNTIME_OVERRIDES = {}

In [ ]:
config = load_experiment_config(CONFIG_PATH, DATA_PATH, overrides=RUNTIME_OVERRIDES)
dataset = validate_dataset(config)
display(pd.DataFrame(dataset["splits"].items(), columns=["split", "rows"]))
display(pd.DataFrame([dataset["train_label_counts"]], index=["train"]))
if dataset["validation_label_counts"] is not None:
    display(pd.DataFrame([dataset["validation_label_counts"]], index=["validation"]))
display(pd.DataFrame(dataset["preview"]))

In [ ]:
assert dataset["labels_are_integers_0_to_4"]
assert set(dataset["train_label_counts"]) == set(range(5))
{
    "columns": dataset["column_mapping"],
    "relevant_fields": dataset["relevant_fields"],
    "official_validation": dataset["official_validation_available"],
}

In [ ]:
manifest = prepare_experiment(config, RUN_ID, resume=True)
{
    key: manifest[key]
    for key in [
        "run_id",
        "dataset_fingerprint",
        "config_hash",
        "runtime_overrides",
        "block_count",
        "heldout_response_count",
        "model_calls_made",
    ]
}

In [ ]:
protocol = inspect_protocol(config, RUN_ID)
assert protocol["zero_prompt_leakage"]
assert all(block["all_orders_same_multiset"] for block in protocol["blocks"])
assert all(block["reverse_random_is_exact_reverse"] for block in protocol["blocks"])
{
    "zero_prompt_leakage": protocol["zero_prompt_leakage"],
    "all_blocks_disjoint": protocol["all_blocks_disjoint"],
    "overlap_events": protocol["calibration"]["overlap_event_count"],
    "heldout_prompts": protocol["heldout_prompt_count"],
    "heldout_responses": protocol["heldout_response_count"],
}

In [ ]:
for block in protocol["blocks"]:
    print(f"Block {block['block_id']} — {block['distinct_prompt_count']} distinct prompts")
    display(pd.DataFrame(block["examples"])[["example_id", "human_score", "prompt", "response"]])

In [ ]:
for block in protocol["blocks"]:
    order_rows = []
    for order_name, entries in block["orders"].items():
        order_rows.extend({"order": order_name, **entry} for entry in entries)
    print(f"Block {block['block_id']} orderings")
    display(pd.DataFrame(order_rows))
    print(
        "same multiset:",
        block["all_orders_same_multiset"],
        "| exact reverse:",
        block["reverse_random_is_exact_reverse"],
    )

In [ ]:
estimate = estimate_experiment(config, RUN_ID)
pd.DataFrame(estimate.items(), columns=["quantity", "value"])